# 仕様書A: Meep版のColab高速化ノートブック (`colab_setup.ipynb`)
**A1: TPUホストのマルチコアMPI ＋ A2: アルゴリズム削減（多点同時励起・減衰停止・クラスタリング解析・STEP 2.5/3 統合）**

各種コードは GitHub リポジトリ [`kyouoku/colab_GPU`](https://github.com/kyouoku/colab_GPU.git) の `spv_colab/` から自動取得します。

---

### パイプライン概要
1. **環境確認 & Google Drive マウント**: `nproc`（物理コア数）、メモリ容量を自動検出し、結果保存先 `OUT_DIR` を Google Drive 上に設定します。
2. **MPI版 PyMeep 導入 (`condacolab`)**: `mamba install -c conda-forge pymeep=*=mpi_mpich_* h5py pandas matplotlib` により MPI 対応 Meep を構築します。
3. **GitHub リポジトリ取得 (`spv_colab/`)**: `https://github.com/kyouoku/colab_GPU.git` をクローン（または `git pull`）して `spv_colab/` モジュール群を読み込みます。
4. **パラメータ設定 (`config.py`)**: 構造種別 `structure_type`（`3`: ディスク、`4`: リング/ラフネスディスク、`5`: ディスク＋導波路、`6`: ボウタイ・ツイスト格子）と探索帯域・精度を設定し、解像度やセルサイズを自動導出します。
5. **MPI並列実行 (`mpirun`)**: 物理コア数 `$NP` で `spv_colab/run_sim.py` を並列実行します（偶 `pz=+1` / 奇 `pz=-1` 両対応、STEP ごとに Google Drive へリアルタイム保存）。
6. **結果表示 & 検証 (§8)**: `summary.csv`、`best condition *.txt`、モードプロファイル画像の表示、および並列効率スケーリング・全構造 draft 検証を実行します。

## 1. ランタイム環境確認 & Google Drive マウント (§2)
TPU v6e-1 / v5e-1 ホストCPU（またはハイメモリCPU）の論理・物理コア数とメモリ容量を確認し、Google Drive をマウントして出力先 `OUT_DIR` を設定します。

In [ ]:
import multiprocessing
import os
import platform
import subprocess

def detect_host_resources():
    logical_cpus = multiprocessing.cpu_count()
    phys_cores = logical_cpus
    cpu_model = platform.processor() or "Unknown CPU"
    try:
        lscpu_out = subprocess.check_output(["lscpu"], text=True)
        for line in lscpu_out.splitlines():
            if "Model name:" in line:
                cpu_model = line.split(":", 1)[1].strip()
        p_out = subprocess.check_output(["lscpu", "-p=CORE,SOCKET"], text=True)
        core_set = {line.strip() for line in p_out.splitlines() if not line.startswith("#") and "," in line}
        if core_set:
            phys_cores = len(core_set)
    except Exception:
        phys_cores = max(1, logical_cpus // 2) if logical_cpus > 1 else 1

    mem_gb = 0.0
    if os.path.exists("/proc/meminfo"):
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal:"):
                    mem_gb = int(line.split()[1]) / (1024 ** 2)
                    break

    print("=" * 72)
    print(f"CPU モデル          : {cpu_model}")
    print(f"論理 CPU 数 (nproc) : {logical_cpus}")
    print(f"物理コア数 (NP推奨) : {phys_cores} (HT除外)")
    print(f"メモリ容量 (RAM)    : {mem_gb:.2f} GB")
    print("=" * 72)
    if phys_cores < 4:
        print("[情報] 物理コア数が少ない環境です。TPU v6e-1 / v5e-1 または ハイメモリCPU ランタイムへの切り替えを推奨します。")
    return phys_cores, logical_cpus, mem_gb

NP_PHYSICAL, NPROC_LOGICAL, HOST_MEM_GB = detect_host_resources()
os.environ["NP"] = str(NP_PHYSICAL)

# Google Drive のマウントと OUT_DIR の設定 (§2, §9)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/spv_colab_results"
except ImportError:
    OUT_DIR = os.path.abspath("./spv_colab_results")

os.makedirs(OUT_DIR, exist_ok=True)
os.environ["SPV_OUT_DIR"] = OUT_DIR
print(f"出力先ディレクトリ (OUT_DIR): {OUT_DIR}")

## 2. `condacolab` と MPI版 PyMeep のインストール (§2)
- 初回実行時は `condacolab.install()` により **自動的にランタイムが再起動** されます。再起動後、もう一度上からセルを実行してください（2回目は自動でスキップされます）。
- 続いて `mamba install -c conda-forge pymeep=*=mpi_mpich_* h5py pandas matplotlib` を実行し、Meep バージョンと MPI を確認します。

In [ ]:
import subprocess
import sys

def setup_condacolab_and_meep():
    # 1. condacolab のチェックと導入
    try:
        import condacolab
        condacolab.check()
        print("[OK] condacolab は既に有効です。")
    except ImportError:
        print("[導入] condacolab をインストールします（完了後にランタイムが自動再起動します）...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "condacolab"])
        import condacolab
        condacolab.install()
        return

    # 2. pymeep (mpi_mpich) のチェックと導入
    try:
        import meep as mp
        print(f"[OK] PyMeep は既にインストールされています (version: {mp.__version__})")
    except ImportError:
        print("[導入] mamba で MPI版 PyMeep (pymeep=*=mpi_mpich_*) をインストール中（数分かかります）...")
        cmd = [
            "mamba", "install", "-y", "-c", "conda-forge",
            "pymeep=*=mpi_mpich_*", "h5py", "pandas", "matplotlib"
        ]
        subprocess.check_call(cmd)
        import meep as mp
        print(f"[完了] PyMeep インストール完了 (version: {mp.__version__})")

    # 3. 起動時ログ用バージョン確認 (§2)
    try:
        mp_ver = subprocess.check_output(
            ["python", "-c", "import meep as mp; print(mp.__version__)"], text=True
        ).strip()
        mpi_ver = subprocess.check_output(["mpirun", "--version"], text=True).splitlines()[0]
        print(f"Meep version : {mp_ver}")
        print(f"MPI runtime  : {mpi_ver}")
    except Exception as e:
        print(f"バージョン確認警告: {e}")

setup_condacolab_and_meep()

## 3. GitHub リポジトリ (`https://github.com/kyouoku/colab_GPU.git`) の取得 (§3)
GitHub リポジトリ [`kyouoku/colab_GPU`](https://github.com/kyouoku/colab_GPU.git) から `spv_colab/` モジュール群を取得します。

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/kyouoku/colab_GPU.git"

if os.path.isdir("spv_colab") and os.path.isfile("spv_colab/run_sim.py"):
    REPO_DIR = os.path.abspath(".")
    print(f"📂 既存のローカルリポジトリを使用します: {REPO_DIR}")
elif os.path.isdir("../spv_colab") and os.path.isfile("../spv_colab/run_sim.py"):
    REPO_DIR = os.path.abspath("..")
    os.chdir(REPO_DIR)
    print(f"📂 親ディレクトリのリポジトリを使用します: {REPO_DIR}")
else:
    REPO_DIR = "/content/colab_GPU" if os.path.isdir("/content") else os.path.abspath("./colab_GPU")
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        print(f"📥 Cloning {REPO_URL} into {REPO_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        print(f"🔄 Pulling latest changes in {REPO_DIR} ...")
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)

SPV_COLAB_DIR = os.path.join(REPO_DIR, "spv_colab")
if SPV_COLAB_DIR not in sys.path:
    sys.path.insert(0, SPV_COLAB_DIR)

print(f"✅ 作業ディレクトリ: {os.getcwd()}")
print(f"✅ spv_colab パス   : {SPV_COLAB_DIR}")

## 4. ユーザー設定パラメータ (`config.cfg`) の編集と自動導出の確認 (§4, §5)
ユーザーが触る設定は `cfg` 辞書の6項目のみです。変更内容は `config.save_runtime_cfg(cfg)` により MPI ワーカープロセスへ自動反映されます。

In [ ]:
import importlib
import config
importlib.reload(config)

# ==============================================================================
# §4. ユーザー入力パラメータの設定
# ==============================================================================
config.cfg.update(
    structure_type = 6,            # 3:microdisk, 4:rough_microdisk, 5:waveguide_disk, 6:singular_bowtie_phc
    structure = dict(
        # type 6: n, a, r_ratio, slab_ratio, N_ring, twist_deg, core_r_ratio, gap_ratio, bowtie_angle
        # type 3-5: n, period, Disk_R, w, Disk_t, Sac_t, Sac_D, Out_D, Gap, Wg_w, Wg_t, Cir_reso, Dev_rate
        n            = 3.4,
        a            = 0.45,
        r_ratio      = 0.26,
        slab_ratio   = 0.50,
        N_ring       = 8,
        twist_deg    = 3.5,
        core_r_ratio = 0.55,
        gap_ratio    = 0.02,
        bowtie_angle = 60.0,
    ),
    f_range   = (0.55, 0.75),      # 探索帯域（Meep単位 a/λ）
    parities  = ["+", "-"],        # 計算するz対称（偶/奇）
    accuracy  = dict(coarse=2, fine=4), # 「最小特徴寸法あたりの格子数」（粗探索/本計算）
    n_modes   = 2,                 # 本計算で扱う最大モード数
    run_mode  = "ALL",             # ALL / HARMINV / MODE（既存ログの再利用）
)

# MPI サブプロセス (run_sim.py) が読み込めるよう設定を保存
saved_cfg_path = config.save_runtime_cfg(config.cfg)
os.environ["SPV_USER_CFG_JSON"] = saved_cfg_path

# §5 自動導出パラメータのプレビュー表示
derived = config.derive_params(config.cfg, verbose=True)
print("=== 自動導出パラメータ一覧 (§5) ===")
for k in [
    "structure_type", "n", "period", "d_min",
    "res_harminv", "res_mode", "fcen", "df", "cutoff",
    "dist_to_pml", "pml_w", "sp", "parities",
    "n_modes", "run_mode", "tmax_coarse", "merge_refine", "n_shots", "out_dir"
]:
    print(f"  {k:<16}: {derived[k]}")

## 5. MPI 並列シミュレーションの実行 (§2, §6)
- `OMP_NUM_THREADS=1` 等のスレッド制御を設定し、`mpirun --allow-run-as-root -np $NP python spv_colab/run_sim.py` で実行します。
- 実行ログはリアルタイムでセルに出力されると同時に、Google Drive 上の `$OUT_DIR/run_log.txt` に逐次保存されます。
- 万一 Colab セッションが途中で切れた場合は、上のセルで `run_mode = 'MODE'` に変更して再実行することで、保存済みの `best condition *.txt` から STEP 2.5 / STEP 3 を再開できます（§9）。

In [ ]:
import os
import subprocess

def run_mpi_simulation(np_procs=None):
    if np_procs is None:
        np_procs = int(os.environ.get("NP", 2))
    out_dir = os.environ.get("SPV_OUT_DIR", os.path.abspath("./spv_colab_results"))
    os.makedirs(out_dir, exist_ok=True)

    env = os.environ.copy()
    env["OMP_NUM_THREADS"] = "1"
    env["OPENBLAS_NUM_THREADS"] = "1"
    env["MKL_NUM_THREADS"] = "1"
    env["VECLIB_MAXIMUM_THREADS"] = "1"
    env["NUMEXPR_NUM_THREADS"] = "1"
    env["SPV_OUT_DIR"] = out_dir

    run_sim_path = os.path.join(SPV_COLAB_DIR, "run_sim.py")
    cmd = [
        "mpirun", "--allow-run-as-root",
        "-np", str(np_procs),
        "python", "-u", run_sim_path
    ]
    print(f"[実行コマンド] {' '.join(cmd)}")
    print(f"[保存先]       {out_dir}")
    print("-" * 80)

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise RuntimeError(f"MPI シミュレーションがエラー終了しました (exit code={ret})")

run_mpi_simulation()

## 6. 計算結果・モードプロファイルの表示 (§7)
Google Drive (`OUT_DIR`) に出力された以下のファイルを集計・可視化します：
- `summary.csv`（`parity, freq, Q, res, elapsed_sec`）
- `best condition {date}.txt`（高解像度補正後）および `best condition {date}_coarse.txt`（粗探索結果）
- 誘電率分布（`epchk*.png`）およびモードプロファイル（`*Dden.png`, `*Hden.png`）

In [ ]:
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

out_dir = os.environ.get("SPV_OUT_DIR", os.path.abspath("./spv_colab_results"))

# 1. summary.csv の表示
summary_path = os.path.join(out_dir, "summary.csv")
if os.path.exists(summary_path):
    display(Markdown("### `summary.csv`"))
    display(pd.read_csv(summary_path))
else:
    print(f"summary.csv がまだ存在しません: {summary_path}")

# 2. best condition ファイルの表示
best_files = sorted(
    [f for f in glob.glob(os.path.join(out_dir, "best condition *.txt")) if not f.endswith("_coarse.txt")],
    key=os.path.getmtime
)
if best_files:
    latest_best = best_files[-1]
    coarse_best = latest_best.replace(".txt", "_coarse.txt")
    display(Markdown(f"### 最新 `best condition` (高解像度補正後): `{os.path.basename(latest_best)}`"))
    display(pd.read_table(latest_best))
    if os.path.exists(coarse_best):
        display(Markdown(f"### 粗探索 `best condition`: `{os.path.basename(coarse_best)}`"))
        display(pd.read_table(coarse_best))

# 3. モードプロファイル PNG と epchk PNG のインライン表示
mode_pngs = sorted(glob.glob(os.path.join(out_dir, "M(*).png")))
epchk_pngs = sorted(glob.glob(os.path.join(out_dir, "epchk*.png")))

def display_png_grid(png_list, title, max_imgs=16, cols=4):
    if not png_list:
        return
    display(Markdown(f"### {title} ({len(png_list)} 枚)"))
    target = png_list[-max_imgs:]
    rows = (len(target) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = [axes] if rows * cols == 1 else axes.flatten()
    for ax, img_path in zip(axes, target):
        img = plt.imread(img_path)
        ax.imshow(img)
        ax.set_title(os.path.basename(img_path), fontsize=8)
        ax.axis("off")
    for ax in axes[len(target):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

display_png_grid(epchk_pngs, "誘電率断面チェック (epchk)", max_imgs=6, cols=3)
display_png_grid(mode_pngs, "モードプロファイル (Dden / Hden)", max_imgs=16, cols=4)

## 7. 検証・ベンチマーク (§8)
仕様書 §8 の合格基準を検証するためのセル群です（必要に応じて個別に実行してください）：
1. **並列効率スケーリング計測**: `NP = 1, 2, 4, ..., nproc(物理コア数)` で STEP 1 を実行し、スケーリングを計測してログとグラフに出力します。
2. **全構造（type 3〜6）draft 一括検証**: `structure_type = 3, 4, 5, 6` をそれぞれ draft 設定（`accuracy = dict(coarse=1.5, fine=3.0)`）で最後まで実行し、偶/奇パリティのモード検出を確認します。

In [ ]:
# [検証1] 並列効率スケーリング計測 (NP = 1, 2, 4, ..., NP_PHYSICAL) (§8)
import os
import time
import subprocess
import pandas as pd
import matplotlib.pyplot as plt

def run_scaling_benchmark(max_np=None):
    if max_np is None:
        max_np = int(os.environ.get("NP", 2))
    np_list = []
    val = 1
    while val < max_np:
        np_list.append(val)
        val *= 2
    if max_np not in np_list:
        np_list.append(max_np)

    bench_dir = os.path.join(os.environ.get("SPV_OUT_DIR", "./spv_colab_results"), "scaling_benchmark")
    os.makedirs(bench_dir, exist_ok=True)

    # ベンチマーク用に HARMINV 粗探索のみ・parities=['+'] の軽量設定を一時実行
    bench_script = os.path.join(bench_dir, "run_bench_item.py")
    with open(bench_script, "w") as f:
        f.write(
            "import os, sys\n"
            f"sys.path.insert(0, r'{SPV_COLAB_DIR}')\n"
            "import config, run_sim\n"
            "c = dict(config.cfg)\n"
            "c['parities'] = ['+']\n"
            "c['accuracy'] = dict(coarse=1.5, fine=3.0)\n"
            "c['run_mode'] = 'HARMINV'\n"
            "c['tmax_coarse'] = 300\n"
            "c['out_dir'] = os.environ['BENCH_CASE_DIR']\n"
            "run_sim.run_pipeline(c)\n"
        )

    records = []
    for np_val in np_list:
        case_dir = os.path.join(bench_dir, f"np_{np_val}")
        os.makedirs(case_dir, exist_ok=True)
        env = os.environ.copy()
        env["OMP_NUM_THREADS"] = "1"
        env["OPENBLAS_NUM_THREADS"] = "1"
        env["MKL_NUM_THREADS"] = "1"
        env["BENCH_CASE_DIR"] = case_dir

        print(f"=== スケーリング計測中: NP={np_val} ===")
        t0 = time.time()
        subprocess.check_call(
            ["mpirun", "--allow-run-as-root", "-np", str(np_val), "python", bench_script],
            env=env
        )
        elapsed = time.time() - t0
        records.append({"NP": np_val, "elapsed_sec": round(elapsed, 2)})
        print(f"  -> NP={np_val}: {elapsed:.2f} sec")

    df_scale = pd.DataFrame(records)
    t1 = df_scale.loc[0, "elapsed_sec"]
    df_scale["speedup"] = (t1 / df_scale["elapsed_sec"]).round(2)
    df_scale["efficiency_pct"] = ((df_scale["speedup"] / df_scale["NP"]) * 100.0).round(1)

    scale_csv = os.path.join(bench_dir, "scaling_results.csv")
    df_scale.to_csv(scale_csv, index=False)
    print("\n[並列効率計測結果]")
    print(df_scale.to_string(index=False))

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(df_scale["NP"], df_scale["speedup"], "o-", label="Measured Speedup")
    ax.plot(df_scale["NP"], df_scale["NP"], "--", color="gray", label="Ideal Linear")
    ax.set_xlabel("MPI Processes (NP)")
    ax.set_ylabel("Speedup")
    ax.set_title("Meep MPI Scaling on Colab Host CPU")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.savefig(os.path.join(bench_dir, "scaling_plot.png"), dpi=150)
    plt.show()
    return df_scale

# 実行する場合は以下のコメントを外してください:
# df_scaling = run_scaling_benchmark()

In [ ]:
# [検証2] 全構造 (structure_type = 3, 4, 5, 6) の draft 設定 (accuracy coarse=1.5) 一括テスト (§8)
import os
import subprocess
import pandas as pd

def run_all_structures_draft_test():
    np_procs = int(os.environ.get("NP", 2))
    base_out = os.path.join(os.environ.get("SPV_OUT_DIR", "./spv_colab_results"), "draft_all_structures")
    os.makedirs(base_out, exist_ok=True)

    f_ranges = {
        3: (2.15, 2.35),
        4: (2.15, 2.35),
        5: (2.15, 2.35),
        6: (0.55, 0.75),
    }

    all_summaries = []
    for stype in [3, 4, 5, 6]:
        st_out = os.path.join(base_out, f"type_{stype}")
        os.makedirs(st_out, exist_ok=True)
        runner_py = os.path.join(st_out, "run_draft.py")
        f_min, f_max = f_ranges[stype]
        with open(runner_py, "w") as f:
            f.write(
                "import os, sys\n"
                f"sys.path.insert(0, r'{SPV_COLAB_DIR}')\n"
                "import config, run_sim\n"
                "c = dict(config.cfg)\n"
                f"c['structure_type'] = {stype}\n"
                "c['structure'] = {}\n"
                f"c['f_range'] = ({f_min}, {f_max})\n"
                "c['parities'] = ['+', '-']\n"
                "c['accuracy'] = dict(coarse=1.5, fine=3.0)\n"
                "c['n_modes'] = 1\n"
                "c['run_mode'] = 'ALL'\n"
                f"c['out_dir'] = r'{st_out}'\n"
                "run_sim.run_pipeline(c)\n"
            )
        print(f"\n================ 検証実行: structure_type = {stype} ================")
        env = os.environ.copy()
        env["OMP_NUM_THREADS"] = "1"
        subprocess.check_call(
            ["mpirun", "--allow-run-as-root", "-np", str(np_procs), "python", "-u", runner_py],
            env=env
        )
        sum_csv = os.path.join(st_out, "summary.csv")
        if os.path.exists(sum_csv):
            df_s = pd.read_csv(sum_csv)
            df_s.insert(0, "structure_type", stype)
            all_summaries.append(df_s)

    if all_summaries:
        final_df = pd.concat(all_summaries, ignore_index=True)
        print("\n=== 全構造 (type 3〜6) Draft 実行結果サマリー ===")
        print(final_df.to_string(index=False))
        return final_df
    return pd.DataFrame()

# 実行する場合は以下のコメントを外してください:
# df_all_types = run_all_structures_draft_test()